# Xarray with browser-backed Icechunk I/O

`ipygis` is a bridge to GIS libraries running in the browser. There are a couple of things to know to have it working:

- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.
- when using the `@earthmover/icechunk` WASM library, the server must send COOP/COEP headers so that `SharedArrayBuffer` is supported in the browser.

In [ ]:
import numpy as np
from ipygis.icechunk import Repository, jupyter_storage
from ipygis.xarray import open_zarr_async, mosaic_async
from ipygis.zarr import asynchronous as zarr

In [ ]:
ds = {}
for t in ["acc", "dir"]:
    storage = jupyter_storage(f"examples/{t}.icechunk")
    repository = await Repository.open_async(
        storage,
        # backend="@earthmover/icechunk",  # "icechunk-js" is the default
        proxy_url="https://my-proxy.david-brochart.workers.dev/",
        virtual_chunk_prefixes=[f"https://data.hydrosheds.org/file/hydrosheds-v2/{t.upper()}/1s/"],
    )
    session = await repository.readonly_session_async("main")
    ds[t] = await open_zarr_async(session.store, mask_and_scale=False)

In [ ]:
ds["acc"]

In [ ]:
ds["dir"]

In [ ]:
# Load only the small arrays describing the geographic origins
mosaic = {}
for t in ["acc", "dir"]:
    origins = await ds[t][["tile_x", "tile_y"]].load_async()
    raster = ds[t]["0"]
    attrs = raster.attrs
    dx, dy, _ = attrs["model_pixel_scale"]
    if (attrs.get("geographic_type") != 4326 or attrs.get("raster_type") != 1
            or dx <= 0 or dy <= 0 or "model_transformation" in attrs):
        raise ValueError("Expected unrotated WGS84 PixelIsArea tiles")
    
    sources = []
    for tile in range(ds[t].sizes["tile"]):
        west = origins.tile_x.values[tile]
        north = origins.tile_y.values[tile]
        source = raster.isel(tile=tile, drop=True).rename(x="longitude", y="latitude")
        source = source.assign_coords(
            longitude=west + (np.arange(source.sizes["longitude"]) + 0.5) * dx,
            latitude=north - (np.arange(source.sizes["latitude"]) + 0.5) * dy,
        )
        sources.append(source)
    
    # Preserve the source nodata value, including 255 for uint8 directions.
    mosaic[t] = await mosaic_async(
        sources, x="longitude", y="latitude", crs="EPSG:4326",
        fill_value=attrs["_FillValue"],
    )

In [ ]:
# Choose a small geographic region inside an available tile for each dataset.
region = {}
for t in ["acc", "dir"]:
    origins = await ds[t][["tile_x", "tile_y"]].load_async()
    west = float(origins.tile_x.values[0])
    north = float(origins.tile_y.values[0])
    region[t] = await mosaic[t].sel(
        longitude=slice(west + 0.01, west + 0.02),
        latitude=slice(north - 0.01, north - 0.02),
    ).load_async()

In [ ]:
region["acc"]

In [ ]:
region["dir"]

In [ ]:
point = {}
for t in ["acc", "dir"]:
    origins = await ds[t][["tile_x", "tile_y"]].load_async()
    west = float(origins.tile_x.values[0])
    north = float(origins.tile_y.values[0])
    point[t] = await mosaic[t].sel(
        longitude=west + 0.015, latitude=north - 0.015, method="nearest",
    ).load_async()

In [ ]:
point["acc"]

In [ ]:
point["dir"]

In [ ]:
# this is DIR's session

group = await zarr.open_group(session.store, mode="r")
array = await group.getitem("0")
array.shape, array.dtype, array.chunks

In [ ]:
result = await array.getitem((10, 100, 200))
result

In [ ]:
# Close after finishing all array reads.
await session.aclose()
await repository.aclose()